In [2]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os
import warnings
import sys

# --- 0. Ignora avvisi ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

print("--- Inizio Script ---")

# --- 1. Definizione dei Path e Setup ---
try:
    BASE_PATH = Path.cwd().parent 
    DATA_DIR = BASE_PATH / 'data'
    OUTPUT_DIR = DATA_DIR / 'all' / 'questionnaire'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    COMBINED_CSV_PATH = OUTPUT_DIR / 'combined_dataset.csv'
    METRICS_CSV_PATH = OUTPUT_DIR / 'accuracy_report.csv'

    print(f"Cartella Base (..): {BASE_PATH.resolve()}")
    print(f"Cartella Dati (input): {DATA_DIR.resolve()}")
    print(f"Cartella Report (output): {OUTPUT_DIR.resolve()}")

except Exception as e:
    print(f"Errore fatale nel setup dei percorsi: {e}")
    sys.exit(1)


# --- 2. Ricerca File, Estrazione e Unione Dati ---
print("\n--- 2. Ricerca ed Estrazione Dati ---")

search_pattern = 'S*/test_1*/opaq/labels.csv'
print(f"Inizio ricerca file con pattern: {DATA_DIR / search_pattern}")

file_paths = list(DATA_DIR.glob(search_pattern))

if not file_paths:
    print("\nATTENZIONE: Nessun file 'labels.csv' trovato.")
    print("Controlla che la struttura delle cartelle sia corretta.")
    print("Lo script verrà interrotto.")
    sys.exit(0)
else:
    print(f"Trovati {len(file_paths)} file 'labels.csv'. Inizio processamento...")

required_cols = ['label_real', 'label_eye', 'label_after']
all_dataframes = []
files_con_errori = 0

for f_path in file_paths:
    try:
        df = pd.read_csv(f_path, usecols=required_cols)
        
        relative_path_parts = f_path.relative_to(DATA_DIR).parts
        df['source_subject'] = relative_path_parts[0]
        df['source_test'] = relative_path_parts[1]
        
        all_dataframes.append(df)
        
    except Exception as e:
        print(f"ERRORE in {f_path}: {e}.")
        files_con_errori += 1

print(f"Processamento file completato. File letti: {len(all_dataframes)}, File con errori: {files_con_errori}")

if all_dataframes:
    master_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"\nDataFrame combinato creato con successo. Totale righe: {len(master_df)}")
    
    try:
        master_df.to_csv(COMBINED_CSV_PATH, index=False)
        print(f"DataFrame completo salvato in: {COMBINED_CSV_PATH}")
    except Exception as e:
        print(f"Errore nel salvataggio del DataFrame combinato: {e}")
        
    print("\nPrime 5 righe del DataFrame combinato:")
    print(master_df.head())
    
else:
    print("\nNessun dato è stato caricato. Lo script non può continuare.")
    sys.exit(0)


# --- 3. Calcolo delle Metriche di Accuratezza ---
print("\n--- 3. Calcolo Metriche di Accuratezza ---")

if 'master_df' in locals() and not master_df.empty:
    
    y_true = master_df['label_real']
    y_pred_eye = master_df['label_eye']
    y_pred_after = master_df['label_after']
    
    labels = [1, 2, 3]
    target_names = ['1_light', '2_medium', '3_heavy']

    print("\n--- 3a. Accuratezza Generale (Overall) ---")
    
    acc_eye = accuracy_score(y_true, y_pred_eye)
    acc_after = accuracy_score(y_true, y_pred_after)
    
    print(f"Accuratezza 'label_eye' (Prima): {acc_eye:.4f}")
    print(f"Accuratezza 'label_after' (Dopo): {acc_after:.4f}")

    # Otteniamo i report come dizionari per un facile accesso
    report_eye_dict = classification_report(
        y_true, y_pred_eye, labels=labels, target_names=target_names, 
        output_dict=True, zero_division=0
    )
    report_after_dict = classification_report(
        y_true, y_pred_after, labels=labels, target_names=target_names, 
        output_dict=True, zero_division=0
    )

    # --- NUOVA SEZIONE: Accuratezza per Singola Label (Recall) ---
    print("\n--- 3b. Accuratezza per Singola Label (Recall) ---")
    print("(Quanti casi di una classe sono stati indovinati correttamente?)")
    
    print("\n'label_eye' (Prima):")
    for label_name in target_names:
        recall_val = report_eye_dict[label_name]['recall']
        print(f"  - Accuratezza (Recall) per '{label_name}': {recall_val:.4f}")

    print("\n'label_after' (Dopo):")
    for label_name in target_names:
        recall_val = report_after_dict[label_name]['recall']
        print(f"  - Accuratezza (Recall) per '{label_name}': {recall_val:.4f}")


    # --- NUOVA SEZIONE: Matrici di Confusione ---
    print("\n--- 3c. Matrici di Confusione Numeriche ---")
    print("(Le righe rappresentano la classe Vera, le colonne la classe Predetta)")

    # Calcoliamo le matrici
    cm_eye = confusion_matrix(y_true, y_pred_eye, labels=labels)
    cm_after = confusion_matrix(y_true, y_pred_after, labels=labels)
    
    # Formattiamo con Pandas per una bella stampa
    df_cm_eye = pd.DataFrame(cm_eye, 
                             index=[f"Vero_{l}" for l in target_names], 
                             columns=[f"Pred_{l}" for l in target_names])
    
    df_cm_after = pd.DataFrame(cm_after, 
                               index=[f"Vero_{l}" for l in target_names], 
                               columns=[f"Pred_{l}" for l in target_names])

    print("\nMatrice di Confusione 'label_eye' (Prima):")
    print(df_cm_eye)
    
    print("\nMatrice di Confusione 'label_after' (Dopo):")
    print(df_cm_after)


    # --- 4. Salvataggio delle Metriche ---
    print("\n--- 4. Salvataggio Report Metriche Completo ---")
    
    try:
        # Trasformiamo i dizionari in DataFrame
        df_report_eye = pd.DataFrame(report_eye_dict).transpose()
        df_report_after = pd.DataFrame(report_after_dict).transpose()
        
        # Uniamo i due report per un confronto affiancato
        df_metrics = pd.concat(
            [df_report_eye, df_report_after], 
            keys=['prediction_EYE', 'prediction_AFTER'], 
            axis=1
        )
        
        # NOTA: L'accuratezza generale è già inclusa come riga 'accuracy'
        # dal classification_report(), quindi non serve aggiungerla manualmente.
        
        # Salviamo il report combinato
        df_metrics.to_csv(METRICS_CSV_PATH)
        
        print(f"Report metriche combinato salvato in: {METRICS_CSV_PATH}")
        print("\nContenuto del report metriche (lo stesso del file CSV):")
        print(df_metrics)
        
    except Exception as e:
        print(f"Errore nel salvataggio del report metriche: {e}")

else:
    print("ERRORE: La variabile 'master_df' non è definita o è vuota.")

print("\n--- Script completato ---")

--- Inizio Script ---
Cartella Base (..): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv
Cartella Dati (input): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data
Cartella Report (output): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\questionnaire

--- 2. Ricerca ed Estrazione Dati ---
Inizio ricerca file con pattern: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S*\test_1*\opaq\labels.csv
Trovati 12 file 'labels.csv'. Inizio processamento...
Processamento file completato. File letti: 12, File con errori: 0

DataFrame combinato creato con successo. Totale righe: 108
DataFrame completo salvato in: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\questionnaire\combined_dataset.csv

Prime 5 righe del DataFrame combinato:
   label_real  label_eye  label_after source_subject source_test
0           2          2            2            S03      test_1
1           3          1            3            S03      test_1
2           1          3            1            S03      test